# XAI in Investment Banking

## Table of Contents

1. Overview
2. Q&As
3. Dependencies
4. Data
5. Model Training
6. Model Evaluation
7. Model Interpretation
8. Summary
9. Exercises

## 1. Overview

Investment banking is a complex field where decisions can have significant financial implications. Machine learning models are increasingly being used to assist in decision-making processes, such as assessing credit risk, predicting market trends, or evaluating potential mergers and acquisitions. However, the black-box nature of many ML models can be problematic in a highly regulated industry that demands transparency. This is where Explainable AI (XAI) comes in, offering insights into model decisions and helping to build trust with stakeholders.

Imagine if we could use machine learning to help predict the success of an IPO or the likelihood of a merger being approved? Wouldn't that be not only ideal but also desirable by everyone in the industry? I think it is, but it's important to keep in mind that a simple message saying "Hey, this IPO will be successful!" won't suffice. Context and a good explanation will go a long way and can also provide the investment bankers with useful information for their clients and regulatory bodies.

## 2. Q&As

When talking to an investment banker who used an algorithm to assess a potential deal, I would most certainly ask the following (at the very least).

1. Why did this particular deal get flagged as high-potential?
2. What factors led to this conclusion?
3. What should be our next steps?

Some potential answers "I" would like to hear if my investment banker was telling me about a high-potential merger:
1. The target company's strong cash flow and complementary product line significantly contributed to flagging this as a high-potential deal. Their EBITDA margin of 25% is well above the industry average of 18%.
2. The combination of strong financials, market synergies, and potential cost savings through consolidation increased the likelihood of this being a successful merger when compared to similar deals in our database.
3. We need to conduct a more thorough due diligence process, including a detailed analysis of potential synergies and integration challenges. We'll also need to prepare a comprehensive valuation model and start drafting the initial merger agreement.

## 3. Dependencies

Here are the packages we will be using in this notebook.

- `scikit-learn`
- `pandas`
- `joblib`
- `matplotlib`
- `alibi`
- `statsmodels`
- `mlserver`

In [ ]:
!pip install scikit-learn pandas joblib matplotlib alibi numpy rich mlserver

## 4. Data

The dataset we'll be using is a synthetic one created to mimic characteristics of successful and unsuccessful mergers and acquisitions (M&A) deals. While it's not from a real-world source, it serves as a starting point for demonstrating how machine learning and XAI could be applied in investment banking.

Why it matters? Machine learning excels at finding patterns in data, and we should use these tools, with appropriate measures in place, to enhance decision-making in finance. Many deals fail due to unforeseen complications, so if there's a way to help investment bankers make more informed decisions while keeping client information safe and complying with regulations, we should be pushing the needle forward.

Description of variables:
- `DealSize` - size of the deal in millions of dollars
- `IndustryMatch` - similarity of industries (0-1)
- `GeographicOverlap` - measure of geographic market overlap (0-1)
- `DebtToEquityRatio` - of the target company
- `CashFlowToDebtRatio` - of the target company
- `ProfitMargin` - of the target company
- `MarketShareCombined` - potential combined market share post-merger
- `RegulatoryRiskScore` - estimated regulatory risk (0-100)
- `CulturalFitScore` - estimated cultural fit between companies (0-100)
- `Success` - target variable (0 = unsuccessful, 1 = successful)

Let's start by loading and evaluating our data.

In [ ]:
from sklearn.model_selection import train_test_split
from rich import print
import pandas as pd
import numpy as np

# Generate synthetic data
np.random.seed(42)
n_samples = 1000

df = pd.DataFrame({
    'DealSize': np.random.uniform(100, 10000, n_samples),
    'IndustryMatch': np.random.uniform(0, 1, n_samples),
    'GeographicOverlap': np.random.uniform(0, 1, n_samples),
    'DebtToEquityRatio': np.random.uniform(0, 2, n_samples),
    'CashFlowToDebtRatio': np.random.uniform(0, 1, n_samples),
    'ProfitMargin': np.random.uniform(-0.1, 0.3, n_samples),
    'MarketShareCombined': np.random.uniform(0, 0.5, n_samples),
    'RegulatoryRiskScore': np.random.uniform(0, 100, n_samples),
    'CulturalFitScore': np.random.uniform(0, 100, n_samples),
})

# Generate target variable based on some rules
df['Success'] = ((df['IndustryMatch'] > 0.7) & 
                 (df['CashFlowToDebtRatio'] > 0.5) & 
                 (df['ProfitMargin'] > 0.1) & 
                 (df['RegulatoryRiskScore'] < 50)).astype(int)

# Add some noise
df['Success'] = df['Success'].mask(np.random.random(n_samples) < 0.1, 1 - df['Success'])

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
y = df['Success']
X = df.drop(['Success'], axis=1).copy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=9)

## 5. Model Training

For this section, we'll use logistic regression due to its high interpretability and ease of use. It's particularly suited for our binary classification task of predicting M&A success.

If you're new to logistic regression, think of it as a classification algorithm used to predict a binary outcome (e.g., successful or unsuccessful merger).

Here's a quick example in the context of M&A:

Suppose you're an investment banker wanting to predict if a merger will be successful (a binary yes/no outcome) based on deal size, industry match, and other variables. Your process might look like this:

1. Convert the output to a probability between 0-1, representing the chance of a successful merger.

2. Use a linear model to combine the input features and calculate a 'score':
    
    $score = Intercept + DealSize * \beta_1 + IndustryMatch * \beta_2 + ...$

3. Convert this score to a probability using the logistic function:
    
    $probability = \frac{1}{(1 + e^{(-score)})}$

4. If probability > 0.5, predict the merger will succeed. Otherwise, predict it will fail.

While simplified, this should give you an intuition for how the method works in practice. 

Now, let's train our model.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

sns.set(rc={'figure.figsize':(11.7,8.27)})

Feel free to experiment with the parameters below.

In [ ]:
lr_cls = LogisticRegression(random_state=0, max_iter=500, verbose=0)

In [ ]:
lr_cls.fit(X_train, y_train)

In [ ]:
lr_cls.coef_

If you don't have the path below, you can create with the following command in the termianl.

```sh
mkdir -p models/diabetes/
```

In [ ]:
model_path = 'models/diabetes/lr_cls_diabetes.pkl'

In [ ]:
joblib.dump(lr_cls, model_path)

In [ ]:
!ls models/diabetes

In [ ]:
lr_cls = joblib.load(model_path)

Let's do a quick sanity check before we move on to thoroughly evaluating our model. For this, we will 
pick a random sample from the test dataset.

In [ ]:
x = X_test.sample(1)
y = y_test[x.index[0]]
print(f"Actual outcome: {'Successful' if y == 1 else 'Unsuccessful'}")
x

In [ ]:
lr_cls.classes_

In [ ]:
proba = lr_cls.predict_proba(x)
print(f"Probability of success: {proba[0][1]:.2f}")
print(f"Model prediction: {'Successful' if proba[0][1] > 0.5 else 'Unsuccessful'}")

In [ ]:
y_pred = lr_cls.predict(X_test)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

In [ ]:
title = 'Confusion matrix for Logistic Regression'
disp = ConfusionMatrixDisplay.from_estimator(
    lr_cls, X_test, y_test, 
    display_labels=['Not Diabetic', 'Diabetic'],
    cmap=plt.cm.Blues, normalize=None
)
disp.ax_.set_title(title);

## 6. Model Evaluation

The first method we'll explore is called Kernel SHAP. In the context of investment banking:

Imagine a merger success prediction model. The inputs are deal size, industry match, geographic overlap, etc. The model predicts how likely the merger is to succeed.

To explain an individual prediction, Kernel SHAP is like asking:

"How much did each factor contribute to the overall success probability?" 

It determines the SHAP value, or impact, of each feature by comparing deals with and without that factor. Industry match may get a high positive SHAP value because it significantly increases success probability. Regulatory risk might have a negative SHAP value since it decreases the chances of success. By summing the SHAP values for all features, Kernel SHAP explains the total predicted success probability. It reveals why the model predicted that particular level of success given those deal characteristics.

This makes the model's behavior more interpretable, which is crucial in investment banking where decisions need to be justified to clients and regulators.

Let's implement `KernelShap`.

In [ ]:
from alibi.explainers import KernelShap

In [ ]:
explainer = KernelShap(lr_cls.predict_proba, task='classification')
explainer

Explainers in Alibi work in the same fashion as estimators in sklearn, that is, they follow the 
`.fit()` and `.predict()` way of doing things so if you are familiar with sklearn, this step will 
feel familiar to you.

In [ ]:
explainer.fit(X_train)

Once we finish creating an explainer, the object we get back gives us a lot of useful information like the one above.

Note that, running an explainer in a large batch of data can be quite compute intensive (depending on the 
explainer of course), so it is good practice to save your models once your code finishes creating them. Let's 
save ours, load it and test it again.

In [ ]:
explainer_path = 'models/diabetes/lr_cls_explainer.pkl'

In [ ]:
joblib.dump(explainer, explainer_path)

In [ ]:
explainer = joblib.load(explainer_path)

In [ ]:
x = X_test.sample(1)
y = y_test[x.index].iloc[0]
print(y)
x

In [ ]:
features = X_train.columns.to_list()
features

As you might have noticed in the metadata returned to us once we trained our model, `KernelShap` is both 
local and global, which means that it can be applied to one or many samples at a time. Let's try it on our 
random sample from above.

In [ ]:
result = explainer.explain(x)

In [ ]:
result.shap_values[0]

In [ ]:
explainer.predictor(x)

What we're interested in is the `shap_values` returned by our explainer. Let's see what these look like.

In [ ]:
result.shap_values

In [ ]:
def plot_importance(feat_imp, feat_names, class_idx):
    df = pd.DataFrame(data=feat_imp, columns=feat_names).sort_values(by=0, axis='columns')
    feat_imp, feat_names = df.values[0], df.columns
    fig, ax = plt.subplots(figsize=(10, 5))
    y_pos = np.arange(len(feat_imp))
    ax.barh(y_pos, feat_imp)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(feat_names, fontsize=15)
    ax.invert_yaxis()
    ax.set_xlabel(f'Feature effects for class {class_idx}', fontsize=15)
    return ax, fig

In [ ]:
import numpy as np

In [ ]:
plot_importance(result.shap_values[1], features, 'Diabetes');

In [ ]:
import shap

In [ ]:
result = explainer.explain(X_train[:100])

In [ ]:
shap.summary_plot(result.shap_values[0], X_train[:100], features);

A positive SHAP value means the feature pushed the output higher. Negative means it pushed the output lower.

It is important to note that, if we train the explainer on a large amount of data (with some compute expenses), the 
explainer would have learned enough about the model globally to locally explain the interactions for new cases.

## 7. Model Interpretation

While Kernel Shap is considered a black-box method, because we chose logistic regression as our model, we can also interrogate each of the coefficients and interpret the results further.

We'll use `statsmodels` to fit a model again because it provides a nice summary table.

In [ ]:
# !pip install 'alibi[shap]'
!pip install 'alibi[ray]'

In [ ]:
import statsmodels.api as sm

In [ ]:
log_model = sm.Logit(y_train, sm.add_constant(X_train))
log_result = log_model.fit()

In [ ]:
print(log_result.summary2())

In the table above we can examine not only the coefficients of each parameter, but also the standard deviation and 
AIC and BIC values of our model.

Because the coefficients are the logarithms of the odds (i.e. the probability of a positive case over 
the probability of a negative case), we can convert them back into exponentials to get a better sense of 
what each value means.

In [ ]:
np.exp(log_result.params).sort_values(ascending=False)

What do these odds mean for a successful M&A deal? It means that the odds of a successful merger increase by a factor of X for each additional unit of Y, provided every other feature stays unchanged.

We need more context for this, which can be achieved with the standard deviation.

In [ ]:
coefs = log_result.params.drop(labels=['const'])
stdv = np.std(X_train, 0)
abs(coefs * stdv).sort_values(ascending=False)

The preceding table can be interpreted as an approximation of risk factors from high to low according to the model. It's also a model-specific feature importance method, and a global one at that (as it was gathered from a group of samples). It tells us how far away from the mean each of these values are.

For instance, if 'IndustryMatch' has the highest absolute value, it suggests that the similarity between the industries of the merging companies is the most important factor in determining the success of the merger, according to our model.

## 8. Summary

1. Explainable AI (XAI) in investment banking provides clear, understandable explanations for ML model predictions, helping bankers and clients develop trust and make informed decisions about complex financial transactions.

2. Kernel SHAP attributes the impact of each input feature (like deal size or industry match) on a model's prediction of M&A success, providing insights into how these factors influence the model's outcomes.

3. Logistic regression estimates the probability of a binary outcome (successful vs unsuccessful merger) by fitting a linear function to input features and applying a logistic transformation.

4. XAI enhances transparency and accountability in investment banking AI systems, where decisions can have significant financial consequences. It helps uncover the reasoning behind complex models, making them accessible to bankers, clients, and regulators.

5. Machine Learning can assist in predicting the success of mergers and acquisitions by analyzing historical data, potentially improving the accuracy and efficiency of deal assessments in investment banking.

## Bonus: Serving our Models

To serve models and explainers together, you can run a server with both models using `mlserver`. 
To do so, run the following command.

```sh
python servers/diabetes/cls_diabetes_service.py
```

You can test that your server is working with the following commands.

In [ ]:
from mlserver.codecs import NumpyCodec
import requests

In [ ]:
x.values, y

In [ ]:
inf_request = {
    'inputs': [
        NumpyCodec.encode_input(name='payload', payload=x.values).dict()
    ]
}
print(inf_request)

Change the name from **classifier** to **explainer** and back to change the endpoint you 
are hitting.

In [ ]:
model = 'diabetes_classifier'
endpoint = f"http://0.0.0.0:8080/v2/models/{model}/infer"
r = requests.post(endpoint, json=inf_request)
r.json()

## 10. Exercises

### Build an Explainer on IPO Success Data

- Create a synthetic dataset mimicking IPO characteristics (e.g., company financials, market conditions, underwriter reputation).
- Train a logistic regression model to predict IPO success (defined as stock price above offering price after 30 days).
- Create a Kernel Shap explainer using Alibi, or use another method like AnchorTabular or CounterFactual.
- Write a short narrative describing the result. 
- If you choose more than one model, compare the explanations - which features do they highlight? How do they differ? 
- Discuss how these insights could be used in practice by investment bankers advising clients on potential IPOs.